# 02 — Exploratory Data Analysis

Explore the feature matrix before modeling. Understand distributions, correlations, and class imbalance.

**This notebook should answer:** Are the features informative? Are there data quality issues? What does the imbalance look like?

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Load Feature Matrix

In [ ]:
from features.build import build_feature_matrix, save_feature_matrix, FEATURE_COLUMNS, LABEL_COLUMN

# Set use_synthetic_tidal=True if real data not yet available
df = build_feature_matrix(use_synthetic_tidal=True)
save_feature_matrix(df)
print(f"Feature matrix: {df.shape}")
display(df[FEATURE_COLUMNS + [LABEL_COLUMN]].describe())

## 2. Class Imbalance

In [ ]:
n_pos = df[LABEL_COLUMN].sum()
n_neg = len(df) - n_pos
ratio = n_neg / n_pos

print(f"Positive (hotspot) cells:  {n_pos:,} ({100*n_pos/len(df):.2f}%)")
print(f"Negative (no hotspot):     {n_neg:,} ({100*n_neg/len(df):.2f}%)")
print(f"Imbalance ratio:           1:{ratio:.0f}")
print()
print("⚠️  A zero-rule classifier (predict all-negative) achieves")
print(f"   {100*n_neg/len(df):.2f}% accuracy. We do NOT report accuracy.")

## 3. Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, col in zip(axes.ravel(), FEATURE_COLUMNS):
    pos = df[df[LABEL_COLUMN]==1][col].dropna()
    neg = df[df[LABEL_COLUMN]==0][col].dropna()
    
    ax.hist(neg, bins=50, alpha=0.5, color='steelblue', density=True, label='No hotspot')
    ax.hist(pos, bins=50, alpha=0.7, color='orange', density=True, label='Hotspot')
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("\nLook for separation between distributions — that indicates predictive power.")

## 4. Feature Correlations

In [ ]:
corr = df[FEATURE_COLUMNS].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im)
ax.set_xticks(range(len(FEATURE_COLUMNS)))
ax.set_yticks(range(len(FEATURE_COLUMNS)))
ax.set_xticklabels(FEATURE_COLUMNS, rotation=45, ha='right')
ax.set_yticklabels(FEATURE_COLUMNS)
for i in range(len(FEATURE_COLUMNS)):
    for j in range(len(FEATURE_COLUMNS)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()
print("\nHigh correlation (|r| > 0.7) between features may indicate redundancy.")

## 5. Spatial Distribution of Label

In [ ]:
from visualization.hotspot_map import plot_hotspot_catalog
from ingest.hotspot_catalog import load_hotspot_catalog

catalog = load_hotspot_catalog()
fig = plot_hotspot_catalog(catalog)
plt.show()

## EDA Summary

**Fill in before proceeding to feature engineering:**

- Class imbalance ratio: ___
- Most predictive feature (by visual separation): ___
- Any highly correlated feature pairs: ___
- Data quality issues found: ___